# رفع ملف CSV إلى ADLS - مشروع SEEK

هذا النوتبوك مبني على لاب "Working with ADLS Gen2 in Python"، بس معدّل بحيث يستخدم:
- بيانات الاتصال الحقيقية لمشروع SEEK (من ملف `.env`)، مو قيم تجريبية مكتوبة بالكود
- الـ Container الفعلي `raw`، والمجلد `ingested_files`

**مهم: شغّلي الخلايا بالترتيب من فوق لتحت، وحدة وحدة، بدون ما تقفزين أي خلية.**

لو صار أي خطأ توثيق (Authentication)، سوي Kernel → Restart وابدئي من الخلية الأولى من جديد.

## الخطوة 1: تثبيت المكتبات المطلوبة (تسوينها مرة وحدة بس)

In [1]:
%pip install azure-storage-file-datalake pandas python-dotenv

Note: you may need to restart the kernel to use updated packages.


## الخطوة 2: قراءة بيانات الاتصال من ملف `.env`

المسار أدناه يشير مباشرة لملف `.env` الموجود جوا مجلد `SEEK_clean` - هذا يضمن إنه يلقاه، بغض النظر عن مكان حفظ هذا النوتبوك.

In [2]:
import os
from dotenv import load_dotenv

load_dotenv(r"C:\Users\Asald\Downloads\SEEK_clean\.env")

account_name = os.getenv("ADLS_ACCOUNT_NAME")
sas_token = os.getenv("ADLS_SAS_TOKEN")

# تشخيص سريع - تأكد فقط إن القيم وصلت (بدون طباعة القيم نفسها)
print("Account Name موجود؟", bool(account_name))
print("SAS Token موجود؟", bool(sas_token))

Account Name موجود؟ True
SAS Token موجود؟ True


⚠️ لازم تطلع لك القيمتين فوق `True` و `True` قبل ما تكملين. لو طلعت `False` بأي وحدة، توقفي وتأكدي من مسار ملف `.env`.

## الخطوة 3: الاتصال بحساب ADLS

In [3]:
from azure.storage.filedatalake import DataLakeServiceClient

service_client = DataLakeServiceClient(
    account_url=f"https://{account_name}.dfs.core.windows.net",
    credential=sas_token
)

print(f"Connected Successfully to ADLS Account {account_name}")

Connected Successfully to ADLS Account ahadfaiz1


## الخطوة 4: تحديد الـ Container والمجلد

نستخدم Container اسمه `raw` (نفسه اللي فيه ملفات عهد)، والمجلد `ingested_files`.

In [39]:
container = "processed"
directory_name = "ingested_files"

file_system_client = service_client.get_file_system_client(container)
directory_client = file_system_client.get_directory_client(directory_name)

## الخطوة 5: رفع ملف الـ CSV

In [ ]:
local_file_path_final = r"C:\Users\Asald\Downloads\SEEK\SEEK\data\processed\validated.csv"
adls_file_name_final = "validated.csv"

data_lake_file_client_final = file_system_client.get_file_client(adls_file_name_final)

with open(local_file_path_final, "rb") as f:
    file_data_final = f.read()

data_lake_file_client_final.upload_data(file_data_final, overwrite=True)
print("Final CSV Uploaded successfully to processed container!")

Final CSV Uploaded successfully to processed container!


## الخطوة 6: التحقق - هل الملف وصل فعلاً للـ Container؟

In [40]:
for path in file_system_client.get_paths():
    if not path.is_directory:
        print(path.name)

final_2026-09-12.csv
final_2026-09-13.csv
final_2026-09-14.csv
final_2026-09-16.csv
rejected.csv
sales/_SUCCESS
sales/part-00000-27d39b57-a7fa-45b0-92d5-7412b916208d-c000.snappy.parquet
tenders_archive.csv
validated.csv


المفروض تشوفين `ingested_files/supermarket_sales.csv` بالقائمة، جنب ملفات JSON الموجودة من قبل. لو طلع، معناها عهد تقدر تشوفه الحين من جهتها.

## الخطوة 7 (اختياري): التأكد إن الملف يُقرأ صح كـ DataFrame

In [10]:
import pandas as pd

downloaded_file = data_lake_file_client.download_file()
downloaded_bytes = downloaded_file.readall()
df = pd.read_csv(pd.io.common.BytesIO(downloaded_bytes))

df.head()

,tender_id,reference_number,tender_name,tender_number,multiple_search,agency_code,branch_id,branch_name,agency_name,tender_id_string,...,current_date,current_date_time,current_time,is_u_g_r_p,ugrp_rfx_url,ugrp_r_f_x_response_u_r_l,source_entity,sector,tender_details_json,has_full_details
0,682754,231039001153,Local and International Media Monitoring and A...,10260044,NaN,NaN,0,Procurement Department - Entertainment Authority,General Entertainment Authority,gJB8qk*@@**ZDZIl38dvjqvvmg==,...,2026-09-16T00:00:00+03:00,2026-09-16T09:48:33.0104308+03:00,00:00:00,False,NaN,NaN,General Entertainment Authority,Other - needs review: General Entertainment Au...,{},False
1,989874,250839016598,Cleanliness of Wadi Al-Dawasir Governorate and...,2026/1,NaN,NaN,0,Wadi Al-Dawasir Governorate Municipality,Riyadh Region Municipality,obMdlWGlbY4u1rTVOIiXnA==,...,2026-09-16T00:00:00+03:00,2026-09-16T09:49:06.4311318+03:00,00:00:00,False,NaN,NaN,Riyadh Region Municipality,Municipal & Housing,{},False
2,1049358,260239007679,Completion of the project of the Faculty of Ap...,26/0023,NaN,NaN,0,Hail University,Hail University,woAohpoVlUFRjgFfZhPVfg==,...,2026-09-16T00:00:00+03:00,2026-09-16T09:49:06.431133+03:00,00:00:00,False,NaN,NaN,Hail University,Education,{},False
3,1071250,260539000892,Supply and installation of chlorine dioxide ga...,204434,NaN,NaN,0,Procurement Management - Services & Projects P...,Saudi Water Authority,ZPc9ARj8iMHZSdm1CO6Inw==,...,2026-09-12T00:00:00+03:00,2026-09-12T22:10:23.0725111+03:00,00:00:00,False,NaN,NaN,Saudi Water Authority,"Energy, Water & Environment",{},False
4,1076036,260539006490,Works and services for the construction of pie...,54/26/53/1,NaN,NaN,0,Tenders & Contracts Management,Ministry of Interior - General Office,JkfbN5E7CJXNxWN*@@**xLUkeg==,...,2026-09-12T00:00:00+03:00,2026-09-12T22:10:56.7422334+03:00,00:00:00,False,NaN,NaN,Ministry of Interior,Other - needs review: Ministry of Interior,{},False
